In [1]:
import os
import torch
import numpy as np
import xarray as xr
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader

from src.dataset import LazyWeatherDataset
from src.preprocessing import flatten_target_dataset, standardize_with_stats, compute_overall_from_daily_stats
from src.models import get_model
from src.shash_torch import Shash

In [8]:
# ---------------------------
# IG routine
# ---------------------------
def integrated_gradients(model, x, baseline, scalar_fn, steps=256):
    """
    x, baseline: [B, C, H, W, T]
    returns IG tensor same shape
    """
    B = x.shape[0]
    device = x.device

    alphas = torch.linspace(0, 1, steps, device=device).view(steps, 1, 1, 1, 1, 1)

    # Create interpolated path: [steps, B, C, H, W, T]
    path = baseline.unsqueeze(0) + alphas * (x.unsqueeze(0) - baseline.unsqueeze(0))

    # Collapse steps into batch: [steps*B, C, H, W, T]
    path = path.view(steps * B, *x.shape[1:])
    path.requires_grad_(True)

    outputs = scalar_fn(model(path))  # [steps*B]
    grads = torch.autograd.grad(outputs.sum(), path)[0]

    # Restore shape: [steps, B, C, H, W, T]
    grads = grads.view(steps, B, *x.shape[1:])

    avg_grads = (grads[:-1] + grads[1:]) / 2
    avg_grads = avg_grads.mean(dim=0)  # [B, C, H, W, T]
    ig = (x - baseline) * avg_grads

    return ig.detach()


# ---------------------------
# SHASH scalar targets
# ---------------------------

def shash_scalars(pred_params, target_idx):
    """
    pred_params: [B, K*4]
    returns dict of scalar tensors [B]
    """
    B = pred_params.shape[0]
    K = pred_params.shape[1] // 4
    params = pred_params.view(B, K, 4)

    # extract parameters for the chosen target
    p = params[:, target_idx, :]  # [B, 4]

    mu = p[:, 0]
    sigma = torch.exp(p[:, 1])
    gamma = p[:, 2]
    tau = torch.exp(p[:, 3])

    # build SHASH distribution
    sh = Shash(torch.stack([mu, sigma, gamma, tau], dim=-1))

    # true distribution mean (not mu!)
    true_mean = sh.mean()
    median = sh.median()

    # tail probabilities from true SHASH CDF
    x_hi = torch.full_like(mu, 2.0)
    x_lo = torch.full_like(mu, -2.0)

    p_gt_2 = 1.0 - sh.cdf(x_hi)
    p_lt_m2 = sh.cdf(x_lo)

    return {
        "mu": mu,                  # location parameter
        "sigma": sigma,            # scale parameter
        "gamma": gamma,
        "tau": tau,
        "mean": true_mean,         # actual distribution mean
        "median": median,
        "p_gt_2": p_gt_2,
        "p_lt_-2": p_lt_m2,
    }


# ---------------------------
# Baseline builder
# ---------------------------

def build_climatology(loader, device):
    total = None
    count = 0

    for xb, _ in loader:
        xb = xb.to(device)
        if total is None:
            total = xb.sum(dim=0, keepdim=True)
        else:
            total += xb.sum(dim=0, keepdim=True)
        count += xb.shape[0]

    return total / count  # [1, C, H, W, T]


# ---------------------------
# Main driver
# ---------------------------

def run_ig_for_model(model_name, level, inputs, targets, stats,
                     batch_size=32, steps=256, latest=False, n_splits=5):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    days = inputs.day.values
    kf = KFold(n_splits=n_splits, shuffle=False)

    overall_stats = compute_overall_from_daily_stats(stats)

    for fold, (train_idx, val_idx) in enumerate(kf.split(days)):

        print(f"\n=== Fold {fold} ===")

        train_days = days[train_idx]
        val_days = days[val_idx]

        X_train = inputs.sel(day=train_days)
        X_val = inputs.sel(day=val_days)

        fold_stats = compute_overall_from_daily_stats(stats.sel(day=train_days))

        conversion_stats = xr.Dataset({
            v: ((fold_stats[v] - overall_stats[v]) / overall_stats[v.replace('_mean', '_std')])
            if v.endswith('_mean')
            else (fold_stats[v] / overall_stats[v])
            for v in fold_stats.data_vars
        })

        X_train_std = standardize_with_stats(X_train, conversion_stats)
        X_val_std = standardize_with_stats(X_val, conversion_stats)

        train_ds = LazyWeatherDataset(
            X_train_std,
            y=flatten_target_dataset(targets.sel(time=train_days)),
            input_dimensions=5
        )

        val_ds = LazyWeatherDataset(
            X_val_std,
            y=flatten_target_dataset(targets.sel(time=val_days)),
            input_dimensions=5
        )

        train_loader = DataLoader(train_ds, batch_size=batch_size)
        val_loader = DataLoader(val_ds, batch_size=1)

        # ---- Build baseline ----
        print("Building baseline climatology...")
        baseline = build_climatology(train_loader, device)

        # ---- Load model ----
        model = get_model(model_name.split('/')[0], next(iter(train_loader))[0].shape[1:], 36, targets=None).to(device)

        model_path = f"models/{model_name}/fold={fold}/" + ("latest.pt" if latest else "best.pt")
        model.load_state_dict(torch.load(model_path, map_location="cpu")["model_state_dict"])
        model.eval()

        # ---- Accumulators ----
        C, H, W, T = baseline.shape[1:]
        K = 9
        scalars = ["mu", "sigma", "gamma", "tau", "mean", "median", "p_gt_2", "p_lt_-2"]

        chan_acc = {s: torch.zeros(K, C) for s in scalars}
        spat_acc = {s: torch.zeros(K, H, W) for s in scalars}
        temp_acc = {s: torch.zeros(K, T) for s in scalars}

        n_days = 0

        # ---- IG loop ----
        for xb, _ in val_loader:
            xb = xb.to(device)

            if n_days % 10 == 0:
                print(n_days)
            n_days += 1

            for target_idx in range(K):

                for scalar_name in scalars:
                    def scalar_fn(out, sn=scalar_name, ti=target_idx):
                        return shash_scalars(out, ti)[sn]

                    ig = integrated_gradients(model, xb, baseline, scalar_fn, steps=steps)[0].detach().cpu()

                    chan_acc[scalar_name][target_idx] += ig.abs().sum(dim=(1, 2, 3))
                    spat_acc[scalar_name][target_idx] += ig.abs().sum(dim=(0, 3))
                    temp_acc[scalar_name][target_idx] += ig.abs().sum(dim=(0, 1, 2))

        # ---- Normalize ----
        for s in scalars:
            chan_acc[s] /= n_days
            spat_acc[s] /= n_days
            temp_acc[s] /= n_days

        # ---- Save ----
        out_dir = f"results/ig/{model_name}/fold_{fold}"
        os.makedirs(out_dir, exist_ok=True)

        torch.save({
            "channel": chan_acc,
            "spatial": spat_acc,
            "temporal": temp_acc
        }, os.path.join(out_dir, "ig_results.pt"))

        print(f"Saved fold {fold}")

In [3]:
inputs = xr.open_zarr("/glade/work/milesep/convective_outlook_ml/train_inputs_slgt_small_glade.zarr")
targets = xr.open_dataset("data/processed_data/train_targets_slgt_new.nc")
stats = xr.open_dataset("data/processed_data/daily_input_stats_slgt_small_glade.nc")

In [9]:
run_ig_for_model(
    model_name='cnn3d_gelu_0_5/level=slgt_small_glade_new/opt=Adam_lr=0.001_batch=8_crit=ShashNLL',
    level='slgt_small_glade_new',
    inputs=inputs,
    targets=targets,
    stats=stats,
    steps=64,
)


=== Fold 0 ===
Building baseline climatology...
0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
Saved fold 0

=== Fold 1 ===
Building baseline climatology...
0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
Saved fold 1

=== Fold 2 ===
Building baseline climatology...
0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
Saved fold 2

=== Fold 3 ===
Bui